# FA 점수체계 자동화 및 검증

이 노트북은 설명용 문서가 아니라 실행용 검증 노트북이다.

자세한 개념은 `FA_학습가이드.md`와 `FA_전략.md`를 먼저 본다.

## 목표

검증 순서:

```text
1. 데이터 로드
2. 한 섹터 선택
3. 점수 분포 확인
4. 상위/하위 종목 확인
5. 결측과 신뢰도 확인
6. 이후 수익률 검증으로 확장
```

운영 목표는 사람이 매번 노트북을 열어 판단하는 것이 아니다. 이 노트북은 자동화 스크립트가 만든 결과를 사람이 필요할 때 점검하는 보조 화면이다.

자동화의 기본 방향은 하방 방어다.

```text
pass    → 신규 결과 사용
warning → 마지막 정상 결과 유지
fail    → 결과 격리 + 마지막 정상 결과 유지
```

즉 애매하거나 실패한 점수를 새로운 리밸런싱 입력으로 밀어 넣지 않는다.

이렇게 정의하는 이유는 FA 점수가 백테스트와 리밸런싱의 입력값이기 때문이다. 새 점수를 못 쓰는 것보다 더 위험한 것은, 신뢰하기 어려운 새 점수를 정상 결과처럼 사용하는 것이다.

전략 문서의 최신 점수 계산 구조는 아래와 같다.

```text
원천 재무값
→ 파생 지표 계산
→ 같은 분기/섹터 내 percentile 변환
→ level_score, change_score, risk_penalty 계산
→ score_confidence 계산
→ fa_score와 fallback 판정
```

이 노트북의 현재 데이터는 연간 임시 랭킹이므로 위 구조를 완전히 구현하지는 않는다. 대신 자동화 검증 화면이 어떤 항목을 확인해야 하는지 먼저 정의한다.

현재는 `company_quarter_fa` 구현 전이므로 기존 `company_sector_rankings_2021_2025.csv`를 임시 검증 데이터로 사용한다.

여기서 보이는 `overall_score`는 새 FA 전략에서 정의한 공식 점수명이 아니다. `overall_score`는 기존 WICS-DART 연간 섹터 랭킹 파이프라인이 만든 임시 종합점수이며, `etl/wics_dart/core/scoring.py`의 `apply_sector_weighted_score()`에서 `growth_score`, `profitability_score`, `stability_score` 등을 섹터별 가중치로 합산해 생성된다. 새 분기 자동화에서 공식적으로 사용할 점수명은 `fa_score`다.

### `overall_score` 정확한 정의

| 항목 | 내용 |
|---|---|
| 컬럼명 | `overall_score` |
| 원천 파일 | `etl/wics_dart/output/company_sector_rankings_2021_2025.csv` |
| 생성 파이프라인 | `etl/wics_dart/pipeline/build_sector_rankings.py`의 `build_rankings()` 실행 결과 |
| 계산 함수 | `etl/wics_dart/core/scoring.py`의 `apply_sector_weighted_score()` |
| 계산 방식 | 기존 하위 점수인 `growth_score`, `profitability_score`, `stability_score` 및 섹터별 보조 점수를 `SECTOR_WEIGHT_CONFIG` 가중치로 합산 |
| 점수 범위 | 0~1 사이의 기존 연간 섹터 상대평가 점수 |
| 현재 노트북에서의 역할 | 새 분기 점수 화면을 만들기 전까지 검증 절차를 시험하기 위한 임시 대체 점수 |
| 운영 자동화에서의 역할 | 없음. 최종 자동화에서는 `fa_score`로 교체해야 함 |

따라서 이 노트북에서 `overall_score`가 보이면 항상 **기존 연간 랭킹 임시 점수**로 읽어야 한다. `overall_score`를 새 FA 전략의 최종 점수로 해석하면 안 된다.

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

# PROJECT_ROOT: 현재 노트북 위치(docs) 기준 프로젝트 루트 경로
PROJECT_ROOT = Path('..').resolve()

# RANKINGS_PATH: 임시 검증에 사용할 기존 연간 FA 섹터 랭킹 CSV 경로
# 이 CSV는 etl/wics_dart/pipeline/build_sector_rankings.py가 생성한다.
# 이 파일 안의 overall_score는 새 분기 FA 점수(fa_score)가 아니라 기존 연간 랭킹 파이프라인 산출물이다.
# overall_score 생성 위치: etl/wics_dart/core/scoring.py의 apply_sector_weighted_score()
RANKINGS_PATH = PROJECT_ROOT / 'etl' / 'wics_dart' / 'output' / 'company_sector_rankings_2021_2025.csv'

# rankings: 기업-연도-섹터별 기존 연간 랭킹 점수와 하위 점수를 담은 원본 검증 데이터
# 주의: rankings['overall_score']는 운영 자동화 목표 점수인 fa_score가 아니다.
rankings = pd.read_csv(RANKINGS_PATH, dtype={'stock_code': str})
rankings.head()

,company_name,company_name_wics,stock_code,corp_code,fiscal_year,wics_large_code,wics_large,wics_index_code,wics_index_name,wics_date,...,cashflow_score,shareholder_return_score,pipeline_event_score,survival_score,cost_control_score,revenue_generation_score,score_model,overall_score_common,overall_score,overall_bucket
0,노바텍,노바텍,285490,1259232.0,2021,G45,IT,G4520,WICS 기술하드웨어와장비,2026-03-27,...,0.923246,1.000000,0.879570,0.582207,0.959449,0.284559,fallback,0.968396,0.968396,top_20%
1,넥스틴,넥스틴,348210,1080252.0,2021,G45,IT,G4530,WICS 반도체와반도체장비,2026-03-27,...,0.822368,0.378495,0.881720,0.719928,0.910097,0.630217,fallback,0.948661,0.948661,top_20%
2,알서포트,알서포트,131370,838962.0,2021,G45,IT,G4510,WICS 소프트웨어와서비스,2026-03-27,...,0.975877,0.604301,0.563441,0.624978,0.984678,0.382849,fallback,0.946031,0.946031,top_20%
3,시큐브,시큐브,131090,402110.0,2021,G45,IT,G4510,WICS 소프트웨어와서비스,2026-03-27,...,0.980263,0.963441,0.640860,0.682690,0.990132,0.290416,fallback,0.920537,0.920537,top_20%
4,프로텍,프로텍,053610,325112.0,2021,G45,IT,G4530,WICS 반도체와반도체장비,2026-03-27,...,0.679825,0.922581,0.088172,0.531390,0.831217,0.339088,fallback,0.920191,0.920191,top_20%


## 한 섹터 선택

처음에는 하나의 섹터만 본다. 여러 섹터를 한 번에 보면 어떤 기준이 문제인지 찾기 어렵다.

In [10]:
# SAMPLE_SECTOR: 이번 검증에서 살펴볼 WICS 대분류 섹터명
SAMPLE_SECTOR = 'IT'

# SAMPLE_YEAR: 이번 검증에서 살펴볼 회계연도
SAMPLE_YEAR = 2025

# sample: 전체 랭킹 중 선택한 섹터와 연도만 필터링한 검증 샘플
sample = rankings[
    (rankings['wics_large'] == SAMPLE_SECTOR)
    & (rankings['fiscal_year'] == SAMPLE_YEAR)
].copy()

sample.shape

(709, 72)

## 점수 분포 확인

좋은 점수체계라면 상위, 중위, 하위가 어느 정도 나뉘어야 한다. 모든 종목이 비슷한 점수에 몰리면 변별력이 부족하다.

In [11]:
# score_summary: 선택한 샘플의 점수 분포와 결측 현황을 요약한 표
score_summary = pd.DataFrame([{
    'sector': SAMPLE_SECTOR,  # 검증 대상 섹터
    'fiscal_year': SAMPLE_YEAR,  # 검증 대상 연도
    'rows': len(sample),  # 샘플에 포함된 전체 기업 수
    'score_count': sample['overall_score'].notna().sum(),  # 기존 연간 랭킹 임시 점수(overall_score)가 계산된 기업 수
    'score_missing': sample['overall_score'].isna().sum(),  # 기존 연간 랭킹 임시 점수(overall_score)가 결측인 기업 수
    'score_mean': sample['overall_score'].mean(),  # 기존 연간 랭킹 임시 점수 평균
    'score_std': sample['overall_score'].std(),  # 기존 연간 랭킹 임시 점수 표준편차, 변별력 확인용
    'score_min': sample['overall_score'].min(),  # 기존 연간 랭킹 임시 점수 최저값
    'score_max': sample['overall_score'].max(),  # 기존 연간 랭킹 임시 점수 최고값
}])

score_summary

,sector,fiscal_year,rows,score_count,score_missing,score_mean,score_std,score_min,score_max
0,IT,2025,709,592,117,0.499776,0.198675,0.027607,0.908636


## 상위/하위 종목 확인

점수가 좋은지 보려면 상위 종목과 하위 종목의 이유가 설명되어야 한다.

In [12]:
# review_columns: 상위/하위 종목을 사람이 검토할 때 함께 볼 핵심 컬럼 목록
review_columns = [
    'company_name',  # 기업명
    'stock_code',  # 종목코드
    'overall_score',  # 기존 연간 섹터 랭킹 파이프라인의 임시 종합점수, 새 공식 fa_score가 아니며 운영 자동화에서 교체 대상
    'overall_bucket',  # 점수 구간 라벨
    'growth_score',  # 성장성 하위 점수
    'profitability_score',  # 수익성 하위 점수
    'stability_score',  # 안정성 하위 점수
    'cashflow_score',  # 현금흐름 하위 점수
    'shareholder_return_score',  # 주주환원 하위 점수
]

# top_10: 기존 연간 랭킹 임시 점수(overall_score) 기준 상위 10개 기업
top_10 = sample.nlargest(10, 'overall_score')[review_columns]
top_10

,company_name,stock_code,overall_score,overall_bucket,growth_score,profitability_score,stability_score,cashflow_score,shareholder_return_score
10312,엔투텍,227950,0.908636,top_20%,0.977941,0.955694,0.792274,0.794168,0.450450
10313,마이크로컨텍솔,098120,0.908424,top_20%,0.919118,0.941025,0.865129,0.859348,0.780180
10314,기가비스,420770,0.898253,top_20%,NaN,0.838099,0.958407,0.993139,0.709910
10315,슈프리마,236200,0.894502,top_20%,0.827206,0.925842,0.930460,0.963979,0.021622
10316,케이씨텍,281820,0.893138,top_20%,NaN,0.861765,0.924512,0.804460,0.881081
10317,파이버프로,368770,0.881106,top_20%,0.876838,0.967575,0.798904,0.730703,0.690090
10318,테스,095610,0.879896,top_20%,0.922794,0.902664,0.814230,0.929674,0.486486
10319,메카로,241770,0.877658,top_20%,0.930147,0.778294,0.924533,0.828473,0.111712
10320,유비쿼스홀딩스,078070,0.874066,top_20%,0.795956,0.861866,0.964377,0.893654,0.956757
10321,ISC,095340,0.871911,top_20%,0.818015,0.909671,0.888048,0.867925,0.805405


In [13]:
# bottom_10: 기존 연간 랭킹 임시 점수(overall_score) 기준 하위 10개 기업
bottom_10 = sample.nsmallest(10, 'overall_score')[review_columns]
bottom_10

,company_name,stock_code,overall_score,overall_bucket,growth_score,profitability_score,stability_score,cashflow_score,shareholder_return_score
10903,풍원정밀,371950,0.027607,bottom_20%,0.047794,0.029083,0.005942,0.109777,0.396396
10902,코이즈,121850,0.028622,bottom_20%,0.033088,0.023116,0.029661,0.039451,NaN
10901,비트맥스,377030,0.043384,bottom_20%,0.088235,0.038523,0.003393,0.049743,0.246847
10900,모아데이타,288980,0.051922,bottom_20%,0.055147,0.061591,0.039026,0.176672,0.216216
10899,아이씨에이치,368600,0.061056,bottom_20%,0.080882,0.060709,0.041576,0.113208,0.472072
10898,한울소재과학,091440,0.061446,bottom_20%,0.112132,0.016210,0.055994,0.018868,0.365766
10897,더블유씨피,393890,0.079166,bottom_20%,0.003676,0.096309,0.137513,0.066895,0.203604
10896,휴맥스,115160,0.084736,bottom_20%,0.091912,0.132614,0.029683,0.475129,0.378378
10895,휴맥스홀딩스,028080,0.085409,bottom_20%,0.095588,0.136047,0.024591,0.461407,0.099099
10894,제이앤티씨,204270,0.093046,bottom_20%,0.058824,0.085363,0.134953,0.284734,0.007207


## 결측과 신뢰도 확인

상위 종목에 결측이 많으면 점수를 그대로 믿으면 안 된다. 그래서 `score_confidence`가 필요하다.

In [14]:
# score_columns: 임시 신뢰도 계산에 사용할 하위 점수 컬럼 목록
# 실제 자동화에서는 score_model별 실제 산식에 들어간 컬럼만 사용해야 한다.
score_columns = [
    'growth_score',  # 성장성 하위 점수
    'profitability_score',  # 수익성 하위 점수
    'stability_score',  # 안정성 하위 점수
    'cashflow_score',  # 현금흐름 하위 점수
    'shareholder_return_score',  # 주주환원 하위 점수
]

# missing_in_top: 상위 10개 기업에서 하위 점수별 결측 개수를 요약한 표
missing_in_top = (
    top_10[score_columns]
    .isna()
    .sum()
    .rename('missing_count_in_top_10')
    .reset_index()
    .rename(columns={'index': 'score_column'})
)

missing_in_top

,score_column,missing_count_in_top_10
0,growth_score,2
1,profitability_score,0
2,stability_score,0
3,cashflow_score,0
4,shareholder_return_score,0


In [15]:
# score_confidence_temp: 임시 점수 신뢰도
# 선택한 score_columns 중 결측이 아닌 하위 점수 비율을 뜻한다.
sample['score_confidence_temp'] = sample[score_columns].notna().mean(axis=1)

sample.nlargest(10, 'overall_score')[
    ['company_name', 'stock_code', 'overall_score', 'score_confidence_temp']
]

,company_name,stock_code,overall_score,score_confidence_temp
10312,엔투텍,227950,0.908636,1.0
10313,마이크로컨텍솔,098120,0.908424,1.0
10314,기가비스,420770,0.898253,0.8
10315,슈프리마,236200,0.894502,1.0
10316,케이씨텍,281820,0.893138,0.8
10317,파이버프로,368770,0.881106,1.0
10318,테스,095610,0.879896,1.0
10319,메카로,241770,0.877658,1.0
10320,유비쿼스홀딩스,078070,0.874066,1.0
10321,ISC,095340,0.871911,1.0


## 자동 판정 정책

운영 자동화에서는 사람이 매번 상위/하위 종목을 보고 판단하지 않는다. 노트북은 자동 판정 결과를 설명하는 보조 화면으로 둔다.

v1 기본 정책은 하방 방어다.

이 정책은 수익 기회를 더 많이 잡기 위한 규칙이 아니라, 검증되지 않은 점수가 다음 단계로 흘러 들어가는 것을 막기 위한 규칙이다.

| 판정 | 기본 행동 |
|---|---|
| `pass` | 신규 결과 사용 |
| `warning` | 마지막 정상 결과 유지 |
| `fail` | 결과 격리 + 마지막 정상 결과 유지 |

검증 변수는 아래처럼 원천 컬럼에서 계산된다.

| 변수 | 계산 출처 | 의미 |
|---|---|---|
| `score_missing_rate` | `sample['overall_score'].isna().mean()` | 기존 연간 랭킹 임시 점수 자체가 생성되지 않은 종목 비율 |
| `score_std` | `sample['overall_score'].std()` | 기존 연간 랭킹 임시 점수 분포의 변별력 |
| `score_confidence_temp` | `score_columns`의 결측 여부 | 점수가 실제 사용 가능한 하위 지표를 얼마나 기반으로 하는지 |
| `top_confidence_min` | 상위 10개 종목의 `score_confidence_temp` 최솟값 | 상위 종목이 결측 때문에 왜곡되지 않았는지 |

현재 `score_confidence_temp`는 임시 계산이다. 실제 자동화에서는 `score_model`별로 실제 산식에 들어간 하위 지표만 사용해야 한다.

아래 셀은 현재 임시 데이터 기준으로 자동 판정 예시를 만든다. 실제 분기 자동화에서는 같은 로직을 스크립트와 설정 파일로 옮긴다.

In [16]:
# validation_config: 자동 판정에 사용하는 임시 검증 기준값 모음
# 실제 분기 자동화에서는 이 값을 설정 파일로 분리한다.
validation_config = {
    'max_score_missing_rate': 0.25,  # 허용할 최대 점수 결측률
    'min_score_std': 0.05,  # 점수 변별력을 인정할 최소 표준편차
    'min_top_score_confidence': 0.70,  # 상위권 종목에 요구할 최소 임시 신뢰도
}

# 변수 출처
# - score_missing_rate: 기존 연간 랭킹 임시 점수(overall_score) 결측률
# - score_std: 기존 연간 랭킹 임시 점수(overall_score) 분포의 표준편차
# - top_confidence_min: 상위 10개 종목의 score_confidence_temp 최솟값
# 실제 분기 자동화에서는 이 기준값을 설정 파일로 분리한다.

# score_missing_rate: 전체 샘플 중 기존 연간 랭킹 임시 점수(overall_score)가 결측인 비율
score_missing_rate = sample['overall_score'].isna().mean()

# score_std: 기존 연간 랭킹 임시 점수(overall_score)의 표준편차, 점수가 한쪽으로 몰렸는지 확인하는 변별력 지표
score_std = sample['overall_score'].std()

# top_confidence_min: 상위 10개 종목 중 가장 낮은 score_confidence_temp
# 상위권이 결측 재정규화로 왜곡됐는지 확인하는 하방 방어 지표
top_confidence_min = (
    sample.nlargest(10, 'overall_score')['score_confidence_temp'].min()
    if sample['overall_score'].notna().any()
    else 0
)

# checks: 각 검증 규칙의 실제 값, 기준값, pass/warning/fail 결과를 담는 표
checks = pd.DataFrame([
    {
        'check': 'score_missing_rate',  # 점수 결측률 검증 항목명
        'value': score_missing_rate,  # 실제 점수 결측률
        'threshold': validation_config['max_score_missing_rate'],  # 허용 가능한 최대 결측률
        'status': 'pass' if score_missing_rate <= validation_config['max_score_missing_rate'] else 'fail',  # 기준 초과 시 fail
    },
    {
        'check': 'score_std',  # 점수 표준편차 검증 항목명
        'value': score_std,  # 실제 점수 표준편차
        'threshold': validation_config['min_score_std'],  # 변별력으로 인정할 최소 표준편차
        'status': 'pass' if score_std >= validation_config['min_score_std'] else 'warning',  # 변별력이 약하면 warning
    },
    {
        'check': 'top_score_confidence_min',  # 상위권 최소 신뢰도 검증 항목명
        'value': top_confidence_min,  # 상위 10개 종목의 최소 임시 신뢰도
        'threshold': validation_config['min_top_score_confidence'],  # 상위권에 요구하는 최소 신뢰도
        'status': 'pass' if top_confidence_min >= validation_config['min_top_score_confidence'] else 'warning',  # 상위권 신뢰도가 낮으면 warning
    },
])

# validation_status: checks 전체를 종합한 최종 자동 판정
# fail이 하나라도 있으면 fail, warning만 있으면 warning, 모두 통과하면 pass
if (checks['status'] == 'fail').any():
    validation_status = 'fail'
elif (checks['status'] == 'warning').any():
    validation_status = 'warning'
else:
    validation_status = 'pass'

# fallback_action: validation_status별 하방 방어 기본 행동
fallback_action = {
    'pass': 'use_new_result',  # 신규 결과를 다음 단계 입력으로 사용
    'warning': 'hold_previous',  # 마지막 정상 결과를 유지
    'fail': 'quarantine_and_hold_previous',  # 이번 결과를 격리하고 마지막 정상 결과 유지
}[validation_status]

# validation_decision: 이번 샘플의 최종 판정과 fallback 행동을 요약한 표
validation_decision = pd.DataFrame([{
    'sector': SAMPLE_SECTOR,  # 검증 대상 섹터
    'fiscal_year': SAMPLE_YEAR,  # 검증 대상 연도
    'validation_status': validation_status,  # 최종 자동 판정
    'fallback_action': fallback_action,  # 판정에 따른 하방 방어 행동
}])

checks, validation_decision

(                      check     value  threshold status
 0        score_missing_rate  0.165021       0.25   pass
 1                 score_std  0.198675       0.05   pass
 2  top_score_confidence_min  0.800000       0.70   pass,
   sector  fiscal_year validation_status fallback_action
 0     IT         2025              pass  use_new_result)

## 이번 샘플의 판단

이 절은 사람이 최종 결정을 내리는 부분이 아니라, 위 자동 판정 결과를 해석하는 부분이다.

2025년 IT 섹터 샘플 참고값:

| 항목 | 값 |
|---|---:|
| 전체 종목 수 | 709 |
| 점수 존재 종목 수 | 592 |
| 평균 점수 | 0.4998 |
| 표준편차 | 0.1987 |
| 최소 점수 | 0.0276 |
| 최대 점수 | 0.9086 |

판단:

- 점수 분포는 상위/하위가 나뉘므로 변별력은 있다.
- 다만 상위 종목에도 일부 하위 점수 결측이 있어 신뢰도 검증이 필요하다.
- 따라서 운영 자동화에서는 검증 규칙에 따라 `pass`, `warning`, `fail`을 자동 판정한다.
- `warning` 또는 `fail`이면 신규 결과를 바로 쓰지 않고 하방 방어 정책을 적용한다.
- 다음 검증은 상위 20%와 하위 20%의 forward return 비교다.

## 다음에 추가할 검증

```text
1. 분기형 company_quarter_fa 생성
2. operating_margin, roe, debt_ratio 같은 파생 지표 계산
3. available_date 기준 시점 검증
4. level_score, change_score, risk_penalty 계산 결과 검증
5. score_model별 실제 사용 지표 기준으로 score_confidence 계산
6. pass/warning/fail 자동 판정 규칙을 설정 파일로 분리
7. warning/fail 시 hold_previous 또는 quarantine 정책 적용
8. 섹터별 상위 20% / 하위 20% 구분
9. 1개월, 3개월, 6개월 forward return 비교
10. 자동화 스크립트의 산출물 점검 화면으로 노트북 역할 축소
11. vectorBT 입력으로 연결
```